# SV Risk Prediction - Raw Features Only
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Retrain SV models using only raw genomic features (no derived scores):
- Remove score/severity features (too predictive)
- Use only raw measurements and biological indicators
- Achieve realistic performance (not 100%)

## Features Removed
- `sv_pathogenicity_score` (derived)
- `gene_impact_score` (derived)
- `size_impact_score` (derived)
- `type_impact_score` (derived)
- `gene_impact_severity` (derived)

## Features Kept (8 raw features)
- `sv_size` (raw measurement)
- `sv_type_class` (categorical)
- `sv_size_category` (categorical)
- `affected_gene_count` (raw count)
- `has_gene_overlap` (boolean)
- `is_multi_gene_sv` (boolean)
- `affects_pharmacogenes` (boolean)
- `affects_omim_genes` (boolean)

---
## 1. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, 
    recall_score, accuracy_score
)
import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("OK Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "data" / "ml" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"

RANDOM_STATE = 42
N_ITER = 20
CV_FOLDS = 3

print("="*80)
print("SV RISK PREDICTION - RAW FEATURES ONLY")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

---
## 2. Load SV Datasets

In [ ]:
print("\nLoading SV datasets...")

with open(DATA_DIR / "sv_train.pkl", 'rb') as f:
    sv_train = pickle.load(f)
    X_train_full, y_train = sv_train['X'], sv_train['y']

with open(DATA_DIR / "sv_validation.pkl", 'rb') as f:
    sv_val = pickle.load(f)
    X_val_full, y_val = sv_val['X'], sv_val['y']

with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_test_full, y_test = sv_test['X'], sv_test['y']

print(f"Train: {X_train_full.shape}")
print(f"Val: {X_val_full.shape}")
print(f"Test: {X_test_full.shape}")
print(f"\nAll features ({len(X_train_full.columns)}): {list(X_train_full.columns)}")

---
## 3. Filter to Raw Features Only

In [ ]:
print("\n" + "="*80)
print("FILTERING TO RAW FEATURES")
print("="*80)

# Define features to REMOVE (derived scores)
score_features = [
    'sv_pathogenicity_score',
    'gene_impact_score',
    'size_impact_score',
    'type_impact_score',
    'gene_impact_severity'
]

# Keep all features except scores
raw_features = [col for col in X_train_full.columns if col not in score_features]

print(f"\nFeatures REMOVED ({len(score_features)}):")
for feat in score_features:
    if feat in X_train_full.columns:
        print(f"  - {feat}")

print(f"\nFeatures KEPT ({len(raw_features)}):")
for feat in raw_features:
    print(f"  - {feat}")

# Filter datasets
X_train = X_train_full[raw_features].copy()
X_val = X_val_full[raw_features].copy()
X_test = X_test_full[raw_features].copy()

print(f"\nNew dataset shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")

---
## 4. Baseline - Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

print("\n" + "="*80)
print("BASELINE: LOGISTIC REGRESSION")
print("="*80)

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train, y_train)

y_val_pred = lr_model.predict(X_val)
y_val_proba = lr_model.predict_proba(X_val)[:, 1]

lr_results = {
    'accuracy': accuracy_score(y_val, y_val_pred),
    'precision': precision_score(y_val, y_val_pred),
    'recall': recall_score(y_val, y_val_pred),
    'f1': f1_score(y_val, y_val_pred),
    'roc_auc': roc_auc_score(y_val, y_val_proba)
}

print(f"\nValidation Performance:")
print(f"  Accuracy:  {lr_results['accuracy']:.4f}")
print(f"  Precision: {lr_results['precision']:.4f}")
print(f"  Recall:    {lr_results['recall']:.4f}")
print(f"  F1-Score:  {lr_results['f1']:.4f}")
print(f"  ROC-AUC:   {lr_results['roc_auc']:.4f}")

---
## 5. Random Forest with Tuning

In [ ]:
print("\n" + "="*80)
print("RANDOM FOREST WITH HYPERPARAMETER TUNING")
print("="*80)

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

print(f"Tuning Random Forest ({N_ITER} iterations)...")
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_params,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring='recall',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_

print(f"OK Tuning complete")
print(f"Best CV recall: {rf_search.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in rf_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate
y_val_pred = rf_model.predict(X_val)
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

rf_results = {
    'accuracy': accuracy_score(y_val, y_val_pred),
    'precision': precision_score(y_val, y_val_pred),
    'recall': recall_score(y_val, y_val_pred),
    'f1': f1_score(y_val, y_val_pred),
    'roc_auc': roc_auc_score(y_val, y_val_proba)
}

print(f"\nValidation Performance:")
print(f"  Accuracy:  {rf_results['accuracy']:.4f}")
print(f"  Precision: {rf_results['precision']:.4f}")
print(f"  Recall:    {rf_results['recall']:.4f}")
print(f"  F1-Score:  {rf_results['f1']:.4f}")
print(f"  ROC-AUC:   {rf_results['roc_auc']:.4f}")

---
## 6. XGBoost with Tuning

In [ ]:
print("\n" + "="*80)
print("XGBOOST WITH HYPERPARAMETER TUNING")
print("="*80)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

print(f"Tuning XGBoost ({N_ITER} iterations)...")
xgb_search = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),
    xgb_params,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring='recall',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

xgb_search.fit(X_train, y_train)
xgb_model = xgb_search.best_estimator_

print(f"OK Tuning complete")
print(f"Best CV recall: {xgb_search.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate
y_val_pred = xgb_model.predict(X_val)
y_val_proba = xgb_model.predict_proba(X_val)[:, 1]

xgb_results = {
    'accuracy': accuracy_score(y_val, y_val_pred),
    'precision': precision_score(y_val, y_val_pred),
    'recall': recall_score(y_val, y_val_pred),
    'f1': f1_score(y_val, y_val_pred),
    'roc_auc': roc_auc_score(y_val, y_val_proba)
}

print(f"\nValidation Performance:")
print(f"  Accuracy:  {xgb_results['accuracy']:.4f}")
print(f"  Precision: {xgb_results['precision']:.4f}")
print(f"  Recall:    {xgb_results['recall']:.4f}")
print(f"  F1-Score:  {xgb_results['f1']:.4f}")
print(f"  ROC-AUC:   {xgb_results['roc_auc']:.4f}")

---
## 7. LightGBM with Tuning

In [ ]:
print("\n" + "="*80)
print("LIGHTGBM WITH HYPERPARAMETER TUNING")
print("="*80)

lgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 50, 70],
    'subsample': [0.6, 0.8, 1.0]
}

print(f"Tuning LightGBM ({N_ITER} iterations)...")
lgb_search = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
    lgb_params,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring='recall',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

lgb_search.fit(X_train, y_train)
lgb_model = lgb_search.best_estimator_

print(f"OK Tuning complete")
print(f"Best CV recall: {lgb_search.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in lgb_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate
y_val_pred = lgb_model.predict(X_val)
y_val_proba = lgb_model.predict_proba(X_val)[:, 1]

lgb_results = {
    'accuracy': accuracy_score(y_val, y_val_pred),
    'precision': precision_score(y_val, y_val_pred),
    'recall': recall_score(y_val, y_val_pred),
    'f1': f1_score(y_val, y_val_pred),
    'roc_auc': roc_auc_score(y_val, y_val_proba)
}

print(f"\nValidation Performance:")
print(f"  Accuracy:  {lgb_results['accuracy']:.4f}")
print(f"  Precision: {lgb_results['precision']:.4f}")
print(f"  Recall:    {lgb_results['recall']:.4f}")
print(f"  F1-Score:  {lgb_results['f1']:.4f}")
print(f"  ROC-AUC:   {lgb_results['roc_auc']:.4f}")

---
## 8. Model Comparison

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON - RAW FEATURES ONLY")
print("="*80)

comparison = pd.DataFrame([
    {'Model': 'Logistic Regression', **lr_results},
    {'Model': 'Random Forest', **rf_results},
    {'Model': 'XGBoost', **xgb_results},
    {'Model': 'LightGBM', **lgb_results}
])

comparison = comparison.sort_values('f1', ascending=False)
print("\n" + comparison.to_string(index=False))

# Save comparison
comparison.to_csv(METRICS_DIR / "sv_raw_features_comparison.csv", index=False)
print("\nOK Comparison saved: sv_raw_features_comparison.csv")

---
## 9. Best Model - Test Set Evaluation

In [ ]:
print("\n" + "="*80)
print("BEST MODEL - TEST SET EVALUATION")
print("="*80)

# Select best model by F1
best_row = comparison.iloc[0]
best_model_name = best_row['Model']

if best_model_name == 'Random Forest':
    best_model = rf_model
elif best_model_name == 'XGBoost':
    best_model = xgb_model
elif best_model_name == 'LightGBM':
    best_model = lgb_model
else:
    best_model = lr_model

print(f"\nSelected: {best_model_name}")
print(f"Validation F1: {best_row['f1']:.4f}")

# Test set evaluation
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

test_results = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'recall': recall_score(y_test, y_test_pred),
    'f1': f1_score(y_test, y_test_pred),
    'roc_auc': roc_auc_score(y_test, y_test_proba)
}

print(f"\nTest Set Performance:")
print(f"  Accuracy:  {test_results['accuracy']:.4f}")
print(f"  Precision: {test_results['precision']:.4f}")
print(f"  Recall:    {test_results['recall']:.4f}")
print(f"  F1-Score:  {test_results['f1']:.4f}")
print(f"  ROC-AUC:   {test_results['roc_auc']:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"  FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")

# Classification report
print(f"\nDetailed Report:")
print(classification_report(y_test, y_test_pred, target_names=['Low-risk', 'High-risk']))

---
## 10. Feature Importance

In [ ]:
print("\nFeature Importance Analysis...")

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_imp = pd.DataFrame({
        'feature': raw_features,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\nFeature Importance (Raw Features):")
    print(feature_imp.to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(feature_imp)), feature_imp['importance'])
    plt.yticks(range(len(feature_imp)), feature_imp['feature'])
    plt.xlabel('Importance')
    plt.title(f'SV Risk - Feature Importance ({best_model_name})\nRaw Features Only')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "29_sv_raw_features_importance.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print("\nOK Feature importance plot saved")

---
## 11. Comparison: Raw vs All Features

In [ ]:
print("\n" + "="*80)
print("COMPARISON: RAW FEATURES VS ALL FEATURES")
print("="*80)

print("\nAll Features (13 features including scores):")
print("  Validation F1: 1.0000 (perfect - too easy)")
print("  Test F1:       1.0000 (perfect - too easy)")
print("  Conclusion:    Score features make task trivial")

print(f"\nRaw Features ({len(raw_features)} features, no scores):")
print(f"  Validation F1: {best_row['f1']:.4f} (realistic)")
print(f"  Test F1:       {test_results['f1']:.4f} (realistic)")
print(f"  Conclusion:    Model learns from genomic patterns")

print("\nRECOMMENDATION:")
print("  Use raw features model for:")
print("    - More credible performance metrics")
print("    - Better understanding of genomic drivers")
print("    - Demonstrates real ML learning (not memorization)")
print("    - Shows biological feature importance")

---
## 12. Save Results

In [ ]:
# Save best model
model_file = MODEL_DIR / "sv_raw_features_best.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nOK Best model saved: {model_file.name}")

# Save summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'approach': 'raw_features_only',
    'features_removed': score_features,
    'features_kept': raw_features,
    'n_features': len(raw_features),
    'best_model': best_model_name,
    'validation_performance': dict(best_row),
    'test_performance': test_results
}

summary_file = METRICS_DIR / "sv_raw_features_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"OK Summary saved: {summary_file.name}")

print("\n" + "="*80)
print("SV RAW FEATURES ANALYSIS COMPLETE")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nKEY TAKEAWAY:")
print(f"  Using only {len(raw_features)} raw genomic features (no derived scores)")
print(f"  achieves F1={test_results['f1']:.4f} - a realistic, credible result.")
print("\nThis demonstrates the model learns biological patterns,")
print("not just memorizes pre-calculated risk scores.")
print("="*80)